# Energy Demand Forecasting - OWID Energy Dataset
## Comprehensive Analysis Pipeline
**Sections:** Data Understanding > Cleaning > Feature Engineering > EDA > Statistical Tests > Clustering > LSTM-TFT Hybrid Model > Evaluation

**Target Variable:** `primary_energy_consumption` (TWh)
**Data:** Our World in Data - Energy Dataset (annual, multi-country)

In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, json
from collections import defaultdict

# Statistical
from scipy import stats, signal
from scipy.stats import kruskal, mannwhitneyu
from scipy.fft import fft, fftfreq
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import STL

# Scikit-learn
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, mean_absolute_error,
                             mean_squared_error, r2_score)

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100

RESULTS_DIR = r'C:\Users\Oumaima\Desktop\Wiame\Resulta'
DATA_DIR = r'C:\Users\Oumaima\Desktop\Wiame'
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Training will be slower.")

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM: 6.44 GB


---
## 1. Data Understanding
### 1.1 Codebook Analysis

In [3]:
codebook = pd.read_csv(os.path.join(DATA_DIR, 'owid-energy-codebook.csv'))
print(f"Codebook: {codebook.shape[0]} variable definitions")
print(f"Columns in codebook: {codebook.columns.tolist()}\n")

# Categorize variables
categories = {
    'Identifiers': ['country', 'year', 'iso_code', 'population', 'gdp'],
    'Consumption (TWh)': [c for c in codebook['column'] if 'consumption' in str(c) and 'change' not in str(c) and 'per_capita' not in str(c) and 'share' not in str(c)],
    'Electricity': [c for c in codebook['column'] if 'elec' in str(c) and 'per_capita' not in str(c) and 'share' not in str(c)],
    'Share (%)': [c for c in codebook['column'] if 'share' in str(c)],
    'Per Capita': [c for c in codebook['column'] if 'per_capita' in str(c)],
    'Change': [c for c in codebook['column'] if 'change' in str(c)],
    'Production': [c for c in codebook['column'] if 'prod' in str(c) and 'change' not in str(c)],
}

for cat, cols in categories.items():
    print(f"{cat}: {len(cols)} variables")

print("\nKey variables for forecasting:")
key_vars = codebook[codebook['column'].isin([
    'primary_energy_consumption', 'electricity_demand', 'energy_per_capita',
    'fossil_fuel_consumption', 'renewables_consumption', 'gdp', 'population'
])][['column', 'title', 'unit']]
print(key_vars.to_string(index=False))

Codebook: 130 variable definitions
Columns in codebook: ['column', 'title', 'description', 'unit', 'source']

Identifiers: 5 variables
Consumption (TWh): 13 variables
Electricity: 17 variables
Share (%): 27 variables
Per Capita: 31 variables
Change: 32 variables
Production: 6 variables

Key variables for forecasting:
                    column                                        title                               unit
                population                                   Population                             people
                       gdp                 Gross domestic product (GDP) international-$ in 2011 prices ($)
        electricity_demand                           Electricity demand               terawatt-hours (TWh)
         energy_per_capita        Primary energy consumption per capita    kilowatt-hours per person (kWh)
   fossil_fuel_consumption Primary energy consumption from fossil fuels               terawatt-hours (TWh)
primary_energy_consumption             

### 1.2 Main Dataset Inspection

In [4]:
df = pd.read_csv(os.path.join(DATA_DIR, 'owid-energy-data.csv'), sep=';')
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Year range: {df['year'].min()} - {df['year'].max()}")
print(f"Unique countries/regions: {df['country'].nunique()}")

print(f"\nData types:")
print(df.dtypes.value_counts())

print(f"\nSample data (first 3 rows):")
df.head(3)

Dataset shape: 23,232 rows x 130 columns
Year range: 1900 - 2025
Unique countries/regions: 314

Data types:
float64    126
object       3
int64        1
Name: count, dtype: int64

Sample data (first 3 rows):


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


### 1.3 Missing Value Analysis

In [5]:
# Convert all numeric columns
for col in df.columns:
    if col not in ['country', 'iso_code']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct}).sort_values('pct', ascending=False)
missing_df = missing_df[missing_df['count'] > 0]

print(f"Columns with missing values: {len(missing_df)} / {df.shape[1]}")
print(f"\nTop 20 by missing %:")
print(missing_df.head(20).to_string())

fig, ax = plt.subplots(figsize=(16, 8))
missing_df.head(30)['pct'].plot(kind='barh', ax=ax, color='coral')
ax.set_xlabel('Missing %')
ax.set_title('Top 30 Columns by Missing Value Percentage')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '01_missing_values.png'), dpi=150, bbox_inches='tight')
plt.show()

Columns with missing values: 128 / 130

Top 20 by missing %:
                                              count    pct
biofuel_cons_change_pct                       21162  91.09
solar_cons_change_pct                         20717  89.17
nuclear_cons_change_pct                       20653  88.90
wind_cons_change_pct                          20434  87.96
other_renewables_cons_change_pct              19244  82.83
electricity_share_energy                      19151  82.43
other_renewables_energy_per_capita            18087  77.85
gas_energy_per_capita                         18087  77.85
fossil_energy_per_capita                      18087  77.85
hydro_energy_per_capita                       18087  77.85
renewables_energy_per_capita                  18087  77.85
solar_energy_per_capita                       18087  77.85
coal_cons_per_capita                          18087  77.85
low_carbon_energy_per_capita                  18087  77.85
wind_energy_per_capita                        18087  7

### 1.4 Identify Usable Countries & Time Ranges

In [6]:
# Separate aggregate regions (no ISO code) from actual countries
aggregates = df[df['iso_code'].isna() | (df['iso_code'] == '')]['country'].unique()
print(f"Aggregate regions excluded: {len(aggregates)}")
print(f"Examples: {list(aggregates[:10])}")

df_countries = df[~df['country'].isin(aggregates)].copy()
print(f"\nFiltered to actual countries: {df_countries.shape}")

# Coverage analysis for primary_energy_consumption
coverage = df_countries.groupby('country').agg(
    total_years=('year', 'count'),
    energy_years=('primary_energy_consumption', 'count'),
    year_min=('year', 'min'),
    year_max=('year', 'max'),
    avg_energy=('primary_energy_consumption', 'mean')
).sort_values('energy_years', ascending=False)

top_coverage = coverage[coverage['energy_years'] >= 25]
print(f"\nCountries with >= 25 years of energy data: {len(top_coverage)}")
print(f"\nTop 20 by coverage:")
print(top_coverage.head(20).to_string())

# Anomaly detection: negative values or extreme outliers
numeric_cols = df_countries.select_dtypes(include=[np.number]).columns
print(f"\n--- Anomaly Detection ---")
for col in ['primary_energy_consumption', 'population', 'gdp']:
    if col in df_countries.columns:
        neg = (df_countries[col] < 0).sum()
        if neg > 0:
            print(f"  {col}: {neg} negative values detected")
        q99 = df_countries[col].quantile(0.99)
        extreme = (df_countries[col] > q99 * 3).sum()
        if extreme > 0:
            print(f"  {col}: {extreme} extreme outliers (>3x P99)")
print("Anomaly check complete.")

Aggregate regions excluded: 94
Examples: ['ASEAN (Ember)', 'Africa', 'Africa (EI)', 'Africa (EIA)', 'Africa (Ember)', 'Africa (Shift)', 'Asia', 'Asia (Ember)', 'Asia Pacific (EI)', 'Asia and Oceania (EIA)']

Filtered to actual countries: (17134, 130)

Countries with >= 25 years of energy data: 214

Top 20 by coverage:
              total_years  energy_years  year_min  year_max   avg_energy
country                                                                 
Spain                 126            60      1900      2025  1196.094717
New Zealand           125            60      1900      2024   187.955767
Egypt                 125            60      1900      2024   526.088633
Ecuador               125            60      1900      2024    99.697033
Ireland               126            60      1900      2025   136.404700
Israel                125            60      1900      2024   176.129383
Saudi Arabia          125            60      1900      2024  1403.132667
Italy                 1

---
## 2. Data Cleaning & Feature Engineering
### 2.1 Data Cleaning

In [7]:
# Select countries with good coverage
selected_countries = top_coverage.index.tolist()
df_clean = df_countries[df_countries['country'].isin(selected_countries)].copy()
print(f"Selected {len(selected_countries)} countries, {df_clean.shape[0]} rows")

# Key columns for energy demand forecasting
key_cols = [
    'country', 'year', 'iso_code', 'population', 'gdp',
    'primary_energy_consumption', 'energy_per_capita', 'energy_per_gdp',
    'electricity_demand', 'electricity_generation', 'electricity_share_energy',
    'fossil_fuel_consumption', 'fossil_share_energy',
    'coal_consumption', 'coal_share_energy',
    'gas_consumption', 'gas_share_energy',
    'oil_consumption', 'oil_share_energy',
    'nuclear_consumption', 'nuclear_share_energy',
    'renewables_consumption', 'renewables_share_energy',
    'hydro_consumption', 'hydro_share_energy',
    'solar_consumption', 'solar_share_energy',
    'wind_consumption', 'wind_share_energy',
    'biofuel_consumption', 'biofuel_share_energy',
    'carbon_intensity_elec', 'greenhouse_gas_emissions',
    'energy_cons_change_pct', 'energy_cons_change_twh',
    'low_carbon_consumption', 'low_carbon_share_energy',
    'fossil_electricity', 'renewables_electricity',
    'per_capita_electricity'
]

available_cols = [c for c in key_cols if c in df_clean.columns]
df_analysis = df_clean[available_cols].copy()
print(f"Selected {len(available_cols)} columns")

# Ensure numeric types
numeric = [c for c in available_cols if c not in ['country', 'iso_code']]
for col in numeric:
    df_analysis[col] = pd.to_numeric(df_analysis[col], errors='coerce')

# Intelligent missing value handling: forward-fill then backward-fill per country
df_analysis = df_analysis.sort_values(['country', 'year'])
for col in numeric:
    if col != 'year':
        df_analysis[col] = df_analysis.groupby('country')[col].transform(
            lambda x: x.ffill().bfill()
        )

# Drop rows where target is still NaN
before = len(df_analysis)
df_analysis = df_analysis.dropna(subset=['primary_energy_consumption'])
print(f"Dropped {before - len(df_analysis)} rows with no target value")
print(f"Clean dataset: {df_analysis.shape}")

remaining_missing = df_analysis[numeric].isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
print(f"\nRemaining missing values (top 10):")
print(remaining_missing.head(10))

# Fill remaining NaN with 0 for share/consumption columns (logical default)
fill_zero_cols = [c for c in numeric if 'share' in c or 'consumption' in c or 'electricity' in c]
for col in fill_zero_cols:
    df_analysis[col] = df_analysis[col].fillna(0)

Selected 214 countries, 16939 rows
Selected 40 columns
Dropped 0 rows with no target value
Clean dataset: (16939, 40)

Remaining missing values (top 10):
renewables_consumption     8658
oil_consumption            8658
biofuel_share_energy       8658
wind_share_energy          8658
wind_consumption           8658
solar_share_energy         8658
solar_consumption          8658
hydro_share_energy         8658
hydro_consumption          8658
renewables_share_energy    8658
dtype: int64


### 2.2 Advanced Feature Engineering

In [8]:
df_fe = df_analysis.sort_values(['country', 'year']).copy()
target = 'primary_energy_consumption'

# 1. LAG FEATURES (1, 2, 3 years)
for lag in [1, 2, 3]:
    df_fe[f'{target}_lag{lag}'] = df_fe.groupby('country')[target].shift(lag)

# 2. ROLLING STATISTICS (3 and 5 year windows)
for window in [3, 5]:
    df_fe[f'{target}_rmean{window}'] = df_fe.groupby('country')[target].transform(
        lambda x: x.rolling(window, min_periods=1).mean()
    )
    df_fe[f'{target}_rstd{window}'] = df_fe.groupby('country')[target].transform(
        lambda x: x.rolling(window, min_periods=1).std().fillna(0)
    )

# 3. ENERGY MIX RATIOS
df_fe['fossil_renewable_ratio'] = df_fe['fossil_fuel_consumption'] / (df_fe['renewables_consumption'] + 1e-6)
df_fe['coal_gas_ratio'] = df_fe['coal_consumption'] / (df_fe['gas_consumption'] + 1e-6)

# 4. INDUSTRIAL PROXY VARIABLES
df_fe['gdp_per_capita'] = df_fe['gdp'] / (df_fe['population'] + 1e-6)
df_fe['energy_intensity'] = df_fe['primary_energy_consumption'] / (df_fe['gdp'] / 1e12 + 1e-6)

# 5. PER CAPITA METRICS
df_fe['fossil_per_capita_twh'] = df_fe['fossil_fuel_consumption'] / (df_fe['population'] / 1e9 + 1e-6)
df_fe['renewable_per_capita_twh'] = df_fe['renewables_consumption'] / (df_fe['population'] / 1e9 + 1e-6)

# 6. SEASONAL / CYCLICAL FEATURES (year-based)
df_fe['decade'] = (df_fe['year'] // 10) * 10
df_fe['years_since_1965'] = df_fe['year'] - 1965
df_fe['decade_sin'] = np.sin(2 * np.pi * (df_fe['year'] % 10) / 10)
df_fe['decade_cos'] = np.cos(2 * np.pi * (df_fe['year'] % 10) / 10)

# 7. GROWTH RATES
df_fe['energy_growth'] = df_fe.groupby('country')[target].pct_change()
df_fe['gdp_growth'] = df_fe.groupby('country')['gdp'].pct_change()

# 8. ELECTRIFICATION RATE
df_fe['electrification_rate'] = df_fe['electricity_generation'] / (df_fe[target] + 1e-6)

# Drop rows missing lag features
df_fe = df_fe.dropna(subset=[f'{target}_lag1'])
# Fill remaining NaN
df_fe = df_fe.fillna(0)
df_fe = df_fe.replace([np.inf, -np.inf], 0)

new_features = [c for c in df_fe.columns if c not in df_analysis.columns]
print(f"Feature-engineered dataset: {df_fe.shape}")
print(f"New features ({len(new_features)}): {new_features}")

Feature-engineered dataset: (16725, 60)
New features (20): ['primary_energy_consumption_lag1', 'primary_energy_consumption_lag2', 'primary_energy_consumption_lag3', 'primary_energy_consumption_rmean3', 'primary_energy_consumption_rstd3', 'primary_energy_consumption_rmean5', 'primary_energy_consumption_rstd5', 'fossil_renewable_ratio', 'coal_gas_ratio', 'gdp_per_capita', 'energy_intensity', 'fossil_per_capita_twh', 'renewable_per_capita_twh', 'decade', 'years_since_1965', 'decade_sin', 'decade_cos', 'energy_growth', 'gdp_growth', 'electrification_rate']


---
## 3. Exploratory Data Analysis (EDA)
### 3.1 Statistical Summary

In [9]:
summary_cols = ['primary_energy_consumption', 'population', 'gdp', 'gdp_per_capita',
                'energy_per_capita', 'fossil_share_energy', 'renewables_share_energy',
                'carbon_intensity_elec', 'electrification_rate']
avail = [c for c in summary_cols if c in df_fe.columns]

desc = df_fe[avail].describe().round(2)
print("DESCRIPTIVE STATISTICS")
print("=" * 80)
print(desc.to_string())

print("\nSKEWNESS:")
print(df_fe[avail].skew().round(3).to_string())
print("\nKURTOSIS:")
print(df_fe[avail].kurtosis().round(3).to_string())

DESCRIPTIVE STATISTICS
       primary_energy_consumption    population           gdp  gdp_per_capita  energy_per_capita  fossil_share_energy  renewables_share_energy  carbon_intensity_elec  electrification_rate
count                    16725.00  1.672500e+04  1.672500e+04        16725.00           16725.00             16725.00                 16725.00               16725.00              16725.00
mean                       516.61  2.740840e+07  3.242855e+10         4994.37           23520.31                43.34                     4.82                 464.04              24338.83
std                       2218.08  1.032564e+08  3.715029e+10         8388.67           37789.07                45.31                    10.46                 264.42             310959.17
min                          0.00  0.000000e+00  0.000000e+00            0.00               0.00                 0.00                     0.00                   0.00                  0.00
25%                          7.81  1.

### 3.2 Correlation Analysis

In [10]:
corr_cols = ['primary_energy_consumption', 'population', 'gdp', 'gdp_per_capita',
             'energy_per_capita', 'fossil_fuel_consumption', 'renewables_consumption',
             'nuclear_consumption', 'coal_consumption', 'gas_consumption', 'oil_consumption',
             'carbon_intensity_elec', 'electrification_rate', 'energy_growth']
avail_corr = [c for c in corr_cols if c in df_fe.columns]

corr = df_fe[avail_corr].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix - Key Energy Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '02_correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

target_corr = corr['primary_energy_consumption'].drop('primary_energy_consumption').sort_values(ascending=False)
print("\nCorrelations with Primary Energy Consumption:")
print(target_corr.round(3).to_string())


Correlations with Primary Energy Consumption:
fossil_fuel_consumption    0.997
oil_consumption            0.939
coal_consumption           0.854
gas_consumption            0.846
renewables_consumption     0.821
nuclear_consumption        0.675
population                 0.567
energy_per_capita          0.156
carbon_intensity_elec      0.072
energy_growth             -0.006
gdp                       -0.011
electrification_rate      -0.018
gdp_per_capita            -0.088


### 3.3 Time Series Visualization

In [11]:
top10 = df_fe.groupby('country')['primary_energy_consumption'].mean().nlargest(10).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Energy consumption trends
ax = axes[0, 0]
for c in top10:
    d = df_fe[df_fe['country'] == c]
    ax.plot(d['year'], d['primary_energy_consumption'], label=c, linewidth=1.5)
ax.set_title('Primary Energy Consumption - Top 10 Countries')
ax.set_xlabel('Year'); ax.set_ylabel('TWh')
ax.legend(fontsize=7, ncol=2)

# Energy mix evolution
ax = axes[0, 1]
mix = ['fossil_share_energy', 'renewables_share_energy', 'nuclear_share_energy']
avail_mix = [c for c in mix if c in df_fe.columns]
yearly_mix = df_fe.groupby('year')[avail_mix].mean()
yearly_mix.plot(ax=ax, linewidth=2)
ax.set_title('Global Average Energy Mix Evolution')
ax.set_ylabel('Share (%)')

# GDP vs Energy
ax = axes[1, 0]
latest = df_fe[df_fe['year'] == df_fe['year'].max()].dropna(subset=['gdp', 'primary_energy_consumption'])
scatter = ax.scatter(latest['gdp'] / 1e12, latest['primary_energy_consumption'],
                     alpha=0.6, s=40, c=latest['renewables_share_energy'], cmap='RdYlGn')
ax.set_xlabel('GDP (Trillion $)'); ax.set_ylabel('Energy Consumption (TWh)')
ax.set_title('GDP vs Energy Consumption (Latest Year)')
plt.colorbar(scatter, ax=ax, label='Renewable Share %')

# Per capita trends
ax = axes[1, 1]
for c in top10[:5]:
    d = df_fe[df_fe['country'] == c]
    if 'energy_per_capita' in d.columns:
        ax.plot(d['year'], d['energy_per_capita'], label=c, linewidth=1.5)
ax.set_title('Energy Per Capita - Top 5 Consumers')
ax.set_xlabel('Year'); ax.set_ylabel('kWh/person')
ax.legend(fontsize=9)

plt.suptitle('Exploratory Data Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '03_time_series_eda.png'), dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Distribution Analysis

In [12]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
dist_vars = ['primary_energy_consumption', 'energy_per_capita', 'gdp_per_capita',
             'fossil_share_energy', 'renewables_share_energy', 'energy_growth']
avail_dist = [v for v in dist_vars if v in df_fe.columns]

for i, var in enumerate(avail_dist[:6]):
    ax = axes[i // 3, i % 3]
    data = df_fe[var].dropna()
    data = data[data.between(data.quantile(0.01), data.quantile(0.99))]
    ax.hist(data, bins=50, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.3)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {data.mean():.1f}')
    ax.axvline(data.median(), color='green', linestyle='--', linewidth=1.5, label=f'Median: {data.median():.1f}')
    ax.set_title(var, fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle('Distribution Analysis (1st-99th percentile)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Advanced Statistical Analysis
### 4.1 Stationarity Tests (ADF & KPSS)

In [13]:
print("=" * 80)
print("STATIONARITY TESTS: ADF & KPSS")
print("=" * 80)

test_countries = df_fe.groupby('country')['primary_energy_consumption'].count().nlargest(8).index.tolist()
stationarity_results = []

for country in test_countries:
    series = df_fe[df_fe['country'] == country]['primary_energy_consumption'].dropna().values
    if len(series) < 15:
        continue

    # ADF Test (H0: unit root / non-stationary)
    adf_stat, adf_p, adf_lags, adf_nobs, adf_crit, _ = adfuller(series, autolag='AIC')

    # KPSS Test (H0: stationary)
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(series, regression='ct', nlags='auto')

    stationarity_results.append({
        'Country': country,
        'ADF Stat': round(adf_stat, 4), 'ADF p': round(adf_p, 4),
        'ADF Result': 'Stationary' if adf_p < 0.05 else 'Non-stationary',
        'KPSS Stat': round(kpss_stat, 4), 'KPSS p': round(kpss_p, 4),
        'KPSS Result': 'Stationary' if kpss_p > 0.05 else 'Non-stationary'
    })

stat_df = pd.DataFrame(stationarity_results)
print(stat_df.to_string(index=False))
print("\nADF: H0=unit root (non-stationary). Reject if p<0.05 -> stationary")
print("KPSS: H0=stationary. Reject if p<0.05 -> non-stationary")

STATIONARITY TESTS: ADF & KPSS
 Country  ADF Stat  ADF p     ADF Result  KPSS Stat  KPSS p    KPSS Result
 Austria   -0.2196 0.9361 Non-stationary     0.3769  0.0100 Non-stationary
 Belgium   -1.0461 0.7361 Non-stationary     0.3127  0.0100 Non-stationary
Bulgaria   -2.0816 0.2520 Non-stationary     0.1791  0.0238 Non-stationary
 Denmark   -1.8876 0.3379 Non-stationary     0.1576  0.0403 Non-stationary
  France   -1.0854 0.7209 Non-stationary     0.2657  0.0100 Non-stationary
 Germany   -1.1025 0.7141 Non-stationary     0.1927  0.0187 Non-stationary
  Greece   -0.4812 0.8956 Non-stationary     0.3081  0.0100 Non-stationary
 Hungary   -1.2647 0.6452 Non-stationary     0.1721  0.0282 Non-stationary

ADF: H0=unit root (non-stationary). Reject if p<0.05 -> stationary
KPSS: H0=stationary. Reject if p<0.05 -> non-stationary


### 4.2 Ljung-Box Test for Autocorrelation

In [14]:
print("=" * 80)
print("LJUNG-BOX TEST FOR AUTOCORRELATION")
print("=" * 80)

lb_results = []
for country in test_countries:
    series = df_fe[df_fe['country'] == country]['primary_energy_consumption'].dropna().values
    if len(series) < 15:
        continue
    lb = acorr_ljungbox(series, lags=[5, 10, 15], return_df=True)
    for lag_val in lb.index:
        lb_results.append({
            'Country': country, 'Lag': lag_val,
            'LB Stat': round(lb.loc[lag_val, 'lb_stat'], 4),
            'p-value': round(lb.loc[lag_val, 'lb_pvalue'], 6),
            'Autocorrelated': 'Yes' if lb.loc[lag_val, 'lb_pvalue'] < 0.05 else 'No'
        })

lb_df = pd.DataFrame(lb_results)
print(lb_df.to_string(index=False))

LJUNG-BOX TEST FOR AUTOCORRELATION
 Country  Lag   LB Stat  p-value Autocorrelated
 Austria    5  589.4409      0.0            Yes
 Austria   10 1086.6885      0.0            Yes
 Austria   15 1475.6358      0.0            Yes
 Belgium    5  580.6138      0.0            Yes
 Belgium   10 1054.1448      0.0            Yes
 Belgium   15 1418.2066      0.0            Yes
Bulgaria    5  576.9967      0.0            Yes
Bulgaria   10 1007.8285      0.0            Yes
Bulgaria   15 1249.9184      0.0            Yes
 Denmark    5  487.4198      0.0            Yes
 Denmark   10  828.4317      0.0            Yes
 Denmark   15 1027.3801      0.0            Yes
  France    5  608.0064      0.0            Yes
  France   10 1136.4906      0.0            Yes
  France   15 1548.3222      0.0            Yes
 Germany    5  572.9297      0.0            Yes
 Germany   10 1028.8691      0.0            Yes
 Germany   15 1351.7977      0.0            Yes
  Greece    5  594.6117      0.0            Yes
  Gre

### 4.3 Spectral Analysis (FFT)

In [15]:
print("=" * 80)
print("SPECTRAL ANALYSIS - Dominant Frequencies")
print("=" * 80)

n_plots = min(6, len(test_countries))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

spectral_results = []
for i, country in enumerate(test_countries[:n_plots]):
    series = df_fe[df_fe['country'] == country]['primary_energy_consumption'].dropna().values
    if len(series) < 10:
        continue

    detrended = signal.detrend(series)
    N = len(detrended)
    yf = fft(detrended)
    xf = fftfreq(N, d=1.0)

    pos = xf > 0
    freqs = xf[pos]
    amplitudes = np.abs(yf[pos]) * 2 / N
    periods = 1.0 / freqs

    ax = axes_flat[i]
    ax.plot(periods, amplitudes, color='navy', linewidth=1.2)
    ax.set_xlabel('Period (years)')
    ax.set_ylabel('Amplitude')
    ax.set_title(f'{country}', fontsize=11)
    ax.set_xlim(2, min(40, N // 2))

    # Find dominant period
    dom_idx = np.argmax(amplitudes)
    dom_period = periods[dom_idx]
    dom_amp = amplitudes[dom_idx]
    ax.axvline(dom_period, color='red', linestyle='--', alpha=0.7)
    ax.annotate(f'{dom_period:.1f}y', (dom_period, dom_amp), fontsize=9, color='red')
    spectral_results.append({'Country': country, 'Dominant Period (years)': round(dom_period, 1),
                             'Amplitude': round(dom_amp, 2)})

# Hide unused subplots
for j in range(n_plots, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Spectral Analysis - Dominant Frequencies in Energy Consumption', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '05_spectral_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print(pd.DataFrame(spectral_results).to_string(index=False))

SPECTRAL ANALYSIS - Dominant Frequencies
 Country  Dominant Period (years)  Amplitude
 Austria                    125.0      47.83
 Belgium                    125.0      65.42
Bulgaria                     62.5      55.32
 Denmark                     31.2      14.92
  France                    125.0     321.87
 Germany                     62.5     315.99


### 4.4 Seasonal Analysis (Kruskal-Wallis & Mann-Whitney)

In [16]:
print("=" * 80)
print("SEASONAL / DECADAL ANALYSIS")
print("=" * 80)

df_fe['decade_group'] = (df_fe['year'] // 10) * 10
decade_groups = [g['primary_energy_consumption'].dropna().values
                 for _, g in df_fe.groupby('decade_group') if len(g) > 10]

if len(decade_groups) >= 2:
    kw_stat, kw_p = kruskal(*decade_groups)
    print(f"Kruskal-Wallis across decades: H={kw_stat:.4f}, p={kw_p:.2e}")
    print(f"Significant difference across decades: {'Yes' if kw_p < 0.05 else 'No'}\n")

# Mann-Whitney post-hoc (consecutive decades)
print("Mann-Whitney post-hoc (consecutive decades):")
decade_labels = sorted(df_fe['decade_group'].dropna().unique())
mw_results = []
for i in range(len(decade_labels) - 1):
    g1 = df_fe[df_fe['decade_group'] == decade_labels[i]]['primary_energy_consumption'].dropna()
    g2 = df_fe[df_fe['decade_group'] == decade_labels[i+1]]['primary_energy_consumption'].dropna()
    if len(g1) > 5 and len(g2) > 5:
        u_stat, u_p = mannwhitneyu(g1, g2, alternative='two-sided')
        sig = '*' if u_p < 0.05 else ''
        mw_results.append({
            'Comparison': f"{int(decade_labels[i])}s vs {int(decade_labels[i+1])}s",
            'U-stat': round(u_stat, 1), 'p-value': round(u_p, 6), 'Significant': sig
        })
        print(f"  {int(decade_labels[i])}s vs {int(decade_labels[i+1])}s: U={u_stat:.1f}, p={u_p:.2e} {sig}")

SEASONAL / DECADAL ANALYSIS
Kruskal-Wallis across decades: H=387.2339, p=1.91e-75
Significant difference across decades: Yes

Mann-Whitney post-hoc (consecutive decades):
  1900s vs 1910s: U=380880.0, p=1.00e+00 
  1910s vs 1920s: U=423200.0, p=1.00e+00 
  1920s vs 1930s: U=423200.0, p=1.00e+00 
  1930s vs 1940s: U=423200.0, p=1.00e+00 
  1940s vs 1950s: U=423200.0, p=1.00e+00 
  1950s vs 1960s: U=449150.0, p=9.87e-01 
  1960s vs 1970s: U=474744.0, p=1.33e-03 *
  1970s vs 1980s: U=1275527.0, p=1.47e-38 *
  1980s vs 1990s: U=1863753.0, p=4.58e-03 *
  1990s vs 2000s: U=2102958.5, p=1.83e-04 *
  2000s vs 2010s: U=2182820.5, p=8.13e-03 *
  2010s vs 2020s: U=992752.0, p=3.10e-04 *


### 4.5 STL Decomposition

In [17]:
print("=" * 80)
print("STL DECOMPOSITION")
print("=" * 80)

stl_country = test_countries[0]
stl_data = df_fe[df_fe['country'] == stl_country].drop_duplicates('year').set_index('year')['primary_energy_consumption'].dropna().sort_index()

print(f"STL on: {stl_country} ({len(stl_data)} years)")

if len(stl_data) >= 14:
    stl = STL(stl_data, period=7, robust=True)
    result = stl.fit()

    fig, axes = plt.subplots(4, 1, figsize=(16, 14))
    result.observed.plot(ax=axes[0], color='black', linewidth=1.5)
    axes[0].set_title(f'Observed - {stl_country}', fontweight='bold')
    result.trend.plot(ax=axes[1], color='blue', linewidth=1.5)
    axes[1].set_title('Trend')
    result.seasonal.plot(ax=axes[2], color='green', linewidth=1.5)
    axes[2].set_title('Seasonal (Cyclical)')
    result.resid.plot(ax=axes[3], color='red', linewidth=1.5)
    axes[3].set_title('Residual')

    for ax in axes:
        ax.set_xlabel('')

    plt.suptitle(f'STL Decomposition - {stl_country}', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, '06_stl_decomposition.png'), dpi=150, bbox_inches='tight')
    plt.show()

    trend_strength = max(0, 1 - result.resid.var() / (result.trend + result.resid).var())
    seasonal_strength = max(0, 1 - result.resid.var() / (result.seasonal + result.resid).var())
    print(f"Trend strength: {trend_strength:.4f}")
    print(f"Seasonal strength: {seasonal_strength:.4f}")
else:
    print(f"Not enough data for STL ({len(stl_data)} < 14)")

STL DECOMPOSITION
STL on: Austria (125 years)
Trend strength: 0.9903
Seasonal strength: 0.0000


### 4.6 PCA on Temporal Energy Profiles

In [18]:
print("=" * 80)
print("PCA ON COUNTRY ENERGY PROFILES")
print("=" * 80)

pca_features = ['energy_per_capita', 'fossil_share_energy', 'renewables_share_energy',
                'nuclear_share_energy', 'coal_share_energy', 'gas_share_energy',
                'oil_share_energy', 'electrification_rate', 'energy_growth']
avail_pca = [c for c in pca_features if c in df_fe.columns]

country_profiles = df_fe.groupby('country')[avail_pca].mean().dropna()
print(f"Country profiles for PCA: {country_profiles.shape}")

scaler_pca = StandardScaler()
X_pca = scaler_pca.fit_transform(country_profiles)

pca = PCA()
X_transformed = pca.fit_transform(X_pca)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scree plot
ax = axes[0]
ax.bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_,
       alpha=0.7, color='steelblue', label='Individual')
cumvar = np.cumsum(pca.explained_variance_ratio_)
ax.plot(range(1, len(cumvar) + 1), cumvar, 'ro-', label='Cumulative')
ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='90% threshold')
ax.set_xlabel('Component'); ax.set_ylabel('Explained Variance Ratio')
ax.set_title('PCA Scree Plot'); ax.legend()

# PC1 vs PC2
ax = axes[1]
ax.scatter(X_transformed[:, 0], X_transformed[:, 1], alpha=0.6, s=40)
p90_x = np.percentile(np.abs(X_transformed[:, 0]), 90)
p90_y = np.percentile(np.abs(X_transformed[:, 1]), 90)
for i, c_name in enumerate(country_profiles.index):
    if abs(X_transformed[i, 0]) > p90_x or abs(X_transformed[i, 1]) > p90_y:
        ax.annotate(c_name, (X_transformed[i, 0], X_transformed[i, 1]), fontsize=7)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA - Country Energy Profiles')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '07_pca_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

loadings = pd.DataFrame(pca.components_[:3].T, columns=['PC1', 'PC2', 'PC3'], index=avail_pca)
print("\nPCA Loadings (top 3 components):")
print(loadings.round(3).to_string())
print(f"\nCumulative variance (3 PCs): {cumvar[2]:.1%}")

PCA ON COUNTRY ENERGY PROFILES
Country profiles for PCA: (214, 9)

PCA Loadings (top 3 components):
                           PC1    PC2    PC3
energy_per_capita        0.195 -0.213  0.566
fossil_share_energy      0.548  0.136 -0.006
renewables_share_energy  0.296 -0.271  0.128
nuclear_share_energy     0.269 -0.365 -0.170
coal_share_energy        0.346 -0.220 -0.329
gas_share_energy         0.364  0.419  0.164
oil_share_energy         0.482  0.133  0.099
electrification_rate     0.030  0.689 -0.197
energy_growth           -0.131  0.113  0.671

Cumulative variance (3 PCs): 58.4%


---
## 5. Clustering & Representative Periods
### 5.1 Cluster Validation

In [19]:
print("=" * 80)
print("CLUSTERING ANALYSIS")
print("=" * 80)

X_cluster = X_transformed[:, :3]
k_range = range(2, 11)
sil, db, ch = [], [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    sil.append(silhouette_score(X_cluster, labels))
    db.append(davies_bouldin_score(X_cluster, labels))
    ch.append(calinski_harabasz_score(X_cluster, labels))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(k_range, sil, 'bo-', linewidth=2)
axes[0].set_title('Silhouette Score (higher=better)'); axes[0].set_xlabel('K')
axes[1].plot(k_range, db, 'ro-', linewidth=2)
axes[1].set_title('Davies-Bouldin Index (lower=better)'); axes[1].set_xlabel('K')
axes[2].plot(k_range, ch, 'go-', linewidth=2)
axes[2].set_title('Calinski-Harabasz Score (higher=better)'); axes[2].set_xlabel('K')
plt.suptitle('Cluster Validation Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '08_cluster_validation.png'), dpi=150, bbox_inches='tight')
plt.show()

optimal_k = list(k_range)[np.argmax(sil)]
print(f"Optimal K (Silhouette): {optimal_k}")
print(f"  Silhouette: {max(sil):.4f}")
print(f"  Davies-Bouldin: {db[optimal_k-2]:.4f}")
print(f"  Calinski-Harabasz: {ch[optimal_k-2]:.4f}")

CLUSTERING ANALYSIS
Optimal K (Silhouette): 4
  Silhouette: 0.7120
  Davies-Bouldin: 0.5820
  Calinski-Harabasz: 223.3664


### 5.2 Final Clustering & Visualization

In [20]:
km_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_cluster)
country_profiles['cluster'] = cluster_labels

fig, ax = plt.subplots(figsize=(14, 9))
colors = plt.cm.Set2(np.linspace(0, 1, optimal_k))
for c_id in range(optimal_k):
    mask = cluster_labels == c_id
    ax.scatter(X_transformed[mask, 0], X_transformed[mask, 1],
               label=f'Cluster {c_id} (n={mask.sum()})', s=70, alpha=0.7, color=colors[c_id])
    for idx, c_name in enumerate(country_profiles.index):
        if mask[idx]:
            ax.annotate(c_name, (X_transformed[idx, 0], X_transformed[idx, 1]),
                       fontsize=6, alpha=0.7)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'Country Clusters (K={optimal_k})')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '09_clusters.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nCluster Profiles (mean values):")
profile = country_profiles.groupby('cluster')[avail_pca].mean()
print(profile.round(3).to_string())

print("\nCountries per cluster:")
for c_id in range(optimal_k):
    countries_in = country_profiles[country_profiles['cluster'] == c_id].index.tolist()
    print(f"  Cluster {c_id}: {countries_in[:10]}{'...' if len(countries_in) > 10 else ''}")


Cluster Profiles (mean values):
         energy_per_capita  fossil_share_energy  renewables_share_energy  nuclear_share_energy  coal_share_energy  gas_share_energy  oil_share_energy  electrification_rate  energy_growth
cluster                                                                                                                                                                                   
0               145882.096                0.000                    0.000                 0.000              0.000             0.000             0.000                 0.167          0.358
1                33291.415               88.024                    9.844                 2.132             20.137            17.168            50.720                 0.532          0.018
2                  526.438               96.923                    3.077                 0.000              6.673            40.538            49.712           2556129.098          0.031
3                12256.058      

### 5.3 Representative Periods & Validation

In [21]:
print("=" * 80)
print("REPRESENTATIVE PERIODS")
print("=" * 80)

representatives = {}
for c_id in range(optimal_k):
    mask = cluster_labels == c_id
    cluster_data = X_cluster[mask]
    centroid = km_final.cluster_centers_[c_id]
    dists = np.linalg.norm(cluster_data - centroid, axis=1)
    closest = np.argmin(dists)
    rep = country_profiles.index[mask][closest]
    representatives[c_id] = rep

print("Representative countries:")
for c_id, rep in representatives.items():
    print(f"  Cluster {c_id}: {rep}")

# Validate representativeness
print("\nRepresentativeness Validation (normalized trends):")
for c_id, rep in representatives.items():
    cluster_countries = country_profiles[country_profiles['cluster'] == c_id].index
    rep_series = df_fe[df_fe['country'] == rep].groupby('year')['primary_energy_consumption'].mean()
    cluster_mean = df_fe[df_fe['country'].isin(cluster_countries)].groupby('year')['primary_energy_consumption'].mean()
    common = rep_series.index.intersection(cluster_mean.index)
    if len(common) > 5:
        r_norm = (rep_series[common] - rep_series[common].mean()) / (rep_series[common].std() + 1e-6)
        c_norm = (cluster_mean[common] - cluster_mean[common].mean()) / (cluster_mean[common].std() + 1e-6)
        rmse = np.sqrt(mean_squared_error(c_norm, r_norm))
        r2 = r2_score(c_norm, r_norm)
        print(f"  Cluster {c_id} ({rep}): RMSE={rmse:.4f}, R2={r2:.4f}")

REPRESENTATIVE PERIODS
Representative countries:
  Cluster 0: United States Virgin Islands
  Cluster 1: Denmark
  Cluster 2: Bangladesh
  Cluster 3: Belize

Representativeness Validation (normalized trends):
  Cluster 0 (United States Virgin Islands): RMSE=0.5058, R2=0.7381
  Cluster 1 (Denmark): RMSE=0.8903, R2=0.2010
  Cluster 2 (Bangladesh): RMSE=0.0000, R2=1.0000
  Cluster 3 (Belize): RMSE=0.2324, R2=0.9447


---
## 6. Deep Learning Model: LSTM + TFT Hybrid
### 6.1 Data Preparation for Modeling

In [22]:

print("MODEL DATA PREPARATION")


feature_cols = [
    'population', 'gdp', 'energy_per_capita', 'energy_per_gdp',
    'fossil_fuel_consumption', 'fossil_share_energy',
    'renewables_consumption', 'renewables_share_energy',
    'nuclear_consumption', 'coal_consumption', 'gas_consumption', 'oil_consumption',
    'carbon_intensity_elec', 'gdp_per_capita',
    'fossil_renewable_ratio', 'electrification_rate',
    'primary_energy_consumption_lag1', 'primary_energy_consumption_lag2',
    'primary_energy_consumption_rmean3', 'primary_energy_consumption_rstd3',
    'decade_sin', 'decade_cos', 'years_since_1965'
]

target_col = 'primary_energy_consumption'
avail_feat = [c for c in feature_cols if c in df_fe.columns]
print(f"Features: {len(avail_feat)}")

model_data = df_fe[['country', 'year'] + avail_feat + [target_col]].replace([np.inf, -np.inf], 0).dropna()
print(f"Model data: {model_data.shape}")

# Scale
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()
X_all = feature_scaler.fit_transform(model_data[avail_feat].values)
y_all = target_scaler.fit_transform(model_data[[target_col]].values)

# Sequences per country
SEQ_LEN = 10

def create_sequences(X, y, countries, years, seq_len):
    X_seq, y_seq, meta = [], [], []
    for country in countries.unique():
        mask = (countries == country).values
        Xc, yc = X[mask], y[mask]
        yrs = years[mask].values
        for i in range(len(Xc) - seq_len):
            X_seq.append(Xc[i:i+seq_len])
            y_seq.append(yc[i+seq_len])
            meta.append({'country': country, 'year': int(yrs[i+seq_len])})
    return np.array(X_seq), np.array(y_seq), meta

X_seq, y_seq, meta = create_sequences(
    X_all, y_all, model_data['country'], model_data['year'], SEQ_LEN)

print(f"Sequences: {X_seq.shape[0]}, Shape: {X_seq.shape[1:]}")

# Time-based split (70/15/15)
years_arr = np.array([m['year'] for m in meta])
sidx = np.argsort(years_arr)
X_seq, y_seq = X_seq[sidx], y_seq[sidx]
meta = [meta[i] for i in sidx]

n = len(X_seq)
t1, t2 = int(0.7 * n), int(0.85 * n)
X_train, y_train = X_seq[:t1], y_seq[:t1]
X_val, y_val = X_seq[t1:t2], y_seq[t1:t2]
X_test, y_test = X_seq[t2:], y_seq[t2:]
meta_test = meta[t2:]

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

class EnergyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BS = 64
train_loader = DataLoader(EnergyDataset(X_train, y_train), batch_size=BS, shuffle=True)
val_loader = DataLoader(EnergyDataset(X_val, y_val), batch_size=BS)
test_loader = DataLoader(EnergyDataset(X_test, y_test), batch_size=BS)
print("DataLoaders ready.")

MODEL DATA PREPARATION
Features: 23
Model data: (16725, 26)
Sequences: 14585, Shape: (10, 23)
Train: 10209 | Val: 2188 | Test: 2188
DataLoaders ready.


### 6.2 Model Architecture (LSTM + Temporal Fusion Transformer Hybrid)

In [23]:
class GatedResidualNetwork(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.elu = nn.ELU()
        self.fc2 = nn.Linear(hidden_dim, out_dim)
        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Linear(out_dim, out_dim)
        self.layer_norm = nn.LayerNorm(out_dim)
        self.skip = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        h = self.elu(self.fc1(x))
        h = self.dropout(self.fc2(h))
        gate = torch.sigmoid(self.gate(h))
        return self.layer_norm(gate * h + (1 - gate) * residual)


class VariableSelectionNetwork(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.grn = GatedResidualNetwork(in_dim, hidden_dim, in_dim, dropout)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        weights = self.softmax(self.grn(x))
        return weights * x, weights


class LSTMTFTHybrid(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, num_heads=4, dropout=0.15):
        super().__init__()
        self.var_select = VariableSelectionNetwork(input_dim, hidden_dim, dropout)
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(hidden_dim)
        self.gate_fc = nn.Linear(hidden_dim, hidden_dim)
        self.output_grn = GatedResidualNetwork(hidden_dim, hidden_dim, hidden_dim, dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        x_sel, weights = self.var_select(x)
        lstm_out, _ = self.lstm(x_sel)
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        attn_out = self.attn_norm(attn_out + lstm_out)
        gate = torch.sigmoid(self.gate_fc(attn_out[:, -1, :]))
        combined = gate * attn_out[:, -1, :] + (1 - gate) * lstm_out[:, -1, :]
        out = self.output_grn(combined)
        return self.head(out)

input_dim = len(avail_feat)
model = LSTMTFTHybrid(input_dim=input_dim, hidden_dim=128, num_layers=2,
                       num_heads=4, dropout=0.15).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: LSTM-TFT Hybrid")
print(f"Input dim: {input_dim}, Hidden: 128, Layers: 2, Heads: 4")
print(f"Total parameters: {total_params:,}")
print(model)

Model: LSTM-TFT Hybrid
Input dim: 23, Hidden: 128, Layers: 2, Heads: 4
Total parameters: 357,998
LSTMTFTHybrid(
  (var_select): VariableSelectionNetwork(
    (grn): GatedResidualNetwork(
      (fc1): Linear(in_features=23, out_features=128, bias=True)
      (elu): ELU(alpha=1.0)
      (fc2): Linear(in_features=128, out_features=23, bias=True)
      (dropout): Dropout(p=0.15, inplace=False)
      (gate): Linear(in_features=23, out_features=23, bias=True)
      (layer_norm): LayerNorm((23,), eps=1e-05, elementwise_affine=True)
      (skip): Identity()
    )
    (softmax): Softmax(dim=-1)
  )
  (lstm): LSTM(23, 128, num_layers=2, batch_first=True, dropout=0.15)
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
  )
  (attn_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (gate_fc): Linear(in_features=128, out_features=128, bias=True)
  (output_grn): GatedResidualNetwork(
    (fc1): Linear(in_featur

### 6.3 Training

In [24]:
criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6)

EPOCHS = 150
PATIENCE = 25
best_val = float('inf')
patience_ctr = 0
hist_train, hist_val = [], []

print("Training LSTM-TFT Hybrid Model")
print("=" * 60)

for epoch in range(EPOCHS):
    # Train
    model.train()
    t_loss = 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
    t_loss /= len(train_loader)
    hist_train.append(t_loss)

    # Validate
    model.eval()
    v_loss = 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            v_loss += criterion(model(Xb), yb).item()
    v_loss /= len(val_loader)
    hist_val.append(v_loss)
    scheduler.step(v_loss)

    if v_loss < best_val:
        best_val = v_loss
        patience_ctr = 0
        torch.save(model.state_dict(), os.path.join(RESULTS_DIR, 'best_model.pth'))
    else:
        patience_ctr += 1

    if (epoch + 1) % 10 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train: {t_loss:.6f} | Val: {v_loss:.6f} | LR: {lr:.2e}")

    if patience_ctr >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest validation loss: {best_val:.6f}")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hist_train, label='Train', linewidth=1.5)
ax.plot(hist_val, label='Validation', linewidth=1.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('Huber Loss')
ax.set_title('Training Curves'); ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '10_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

Training LSTM-TFT Hybrid Model
Epoch  10/150 | Train: 0.000068 | Val: 0.000368 | LR: 1.00e-03
Epoch  20/150 | Train: 0.000056 | Val: 0.000294 | LR: 1.00e-03
Epoch  30/150 | Train: 0.000042 | Val: 0.000326 | LR: 5.00e-04
Epoch  40/150 | Train: 0.000043 | Val: 0.000225 | LR: 5.00e-04
Epoch  50/150 | Train: 0.000044 | Val: 0.000265 | LR: 5.00e-04
Epoch  60/150 | Train: 0.000035 | Val: 0.000204 | LR: 2.50e-04
Epoch  70/150 | Train: 0.000032 | Val: 0.000226 | LR: 2.50e-04
Epoch  80/150 | Train: 0.000029 | Val: 0.000225 | LR: 1.25e-04
Epoch  90/150 | Train: 0.000028 | Val: 0.000217 | LR: 6.25e-05
Epoch 100/150 | Train: 0.000037 | Val: 0.000194 | LR: 6.25e-05
Epoch 110/150 | Train: 0.000030 | Val: 0.000208 | LR: 3.13e-05
Epoch 120/150 | Train: 0.000026 | Val: 0.000192 | LR: 1.56e-05
Epoch 130/150 | Train: 0.000024 | Val: 0.000198 | LR: 7.81e-06
Early stopping at epoch 132

Best validation loss: 0.000190


---
## 7. Model Evaluation
### 7.1 Metrics Computation

In [25]:
model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, 'best_model.pth'), map_location=device))
model.eval()

def evaluate_set(loader, name):
    preds_list, targets_list = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            p = model(Xb.to(device)).cpu().numpy()
            preds_list.append(p)
            targets_list.append(yb.numpy())
    preds = np.concatenate(preds_list).flatten()
    targets = np.concatenate(targets_list).flatten()
    preds_inv = target_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
    targets_inv = target_scaler.inverse_transform(targets.reshape(-1, 1)).flatten()

    mae = mean_absolute_error(targets_inv, preds_inv)
    mape = np.mean(np.abs((targets_inv - preds_inv) / (targets_inv + 1e-8))) * 100
    mse = mean_squared_error(targets_inv, preds_inv)
    rmse = np.sqrt(mse)
    r2 = r2_score(targets_inv, preds_inv)

    print(f"\n{name}: MAE={mae:.2f} TWh | MAPE={mape:.2f}% | RMSE={rmse:.2f} TWh | R2={r2:.4f}")
    return preds_inv, targets_inv, {'MAE': mae, 'MAPE': mape, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

print("=" * 80)
print("MODEL EVALUATION")
print("=" * 80)

tr_p, tr_t, tr_m = evaluate_set(train_loader, "Train")
va_p, va_t, va_m = evaluate_set(val_loader, "Validation")
te_p, te_t, te_m = evaluate_set(test_loader, "Test")

metrics_df = pd.DataFrame({
    'Metric': ['MAE (TWh)', 'MAPE (%)', 'MSE', 'RMSE (TWh)', 'R2'],
    'Train': [tr_m['MAE'], tr_m['MAPE'], tr_m['MSE'], tr_m['RMSE'], tr_m['R2']],
    'Validation': [va_m['MAE'], va_m['MAPE'], va_m['MSE'], va_m['RMSE'], va_m['R2']],
    'Test': [te_m['MAE'], te_m['MAPE'], te_m['MSE'], te_m['RMSE'], te_m['R2']]
})
print("\n" + "=" * 80)
print(metrics_df.round(4).to_string(index=False))

MODEL EVALUATION

Train: MAE=92.07 TWh | MAPE=4848358800.00% | RMSE=222.94 TWh | R2=0.9855

Validation: MAE=182.26 TWh | MAPE=382069800.00% | RMSE=967.08 TWh | R2=0.8703

Test: MAE=271.20 TWh | MAPE=2563021200.00% | RMSE=1678.94 TWh | R2=0.7786

    Metric        Train   Validation         Test
 MAE (TWh) 9.207040e+01 1.822566e+02 2.711990e+02
  MAPE (%) 4.848359e+09 3.820698e+08 2.563021e+09
       MSE 4.970016e+04 9.352453e+05 2.818848e+06
RMSE (TWh) 2.229353e+02 9.670808e+02 1.678943e+03
        R2 9.855000e-01 8.703000e-01 7.786000e-01


### 7.2 Prediction Plots

In [26]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (p, t, name) in zip(axes, [(tr_p, tr_t, 'Train'), (va_p, va_t, 'Validation'), (te_p, te_t, 'Test')]):
    ax.scatter(t, p, alpha=0.3, s=15)
    lo, hi = min(t.min(), p.min()), max(t.max(), p.max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='Perfect')
    ax.set_xlabel('Actual (TWh)'); ax.set_ylabel('Predicted (TWh)')
    ax.set_title(f'{name} (R2={r2_score(t, p):.3f})'); ax.legend()
plt.suptitle('Actual vs Predicted Energy Consumption', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '11_predictions_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

# Test time series
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(range(len(te_t)), te_t, label='Actual', linewidth=1.5, alpha=0.8)
ax.plot(range(len(te_p)), te_p, label='Predicted', linewidth=1.5, alpha=0.8)
ax.fill_between(range(len(te_t)), te_t, te_p, alpha=0.15, color='red')
ax.set_xlabel('Sample'); ax.set_ylabel('TWh')
ax.set_title('Test Set: Actual vs Predicted'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '12_test_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

### 7.3 Save All Outputs

In [27]:
# Save predictions
pred_df = pd.DataFrame({
    'Country': [m['country'] for m in meta_test],
    'Year': [m['year'] for m in meta_test],
    'Actual_TWh': te_t, 'Predicted_TWh': te_p,
    'Error_TWh': te_t - te_p,
    'APE_pct': np.abs((te_t - te_p) / (te_t + 1e-8)) * 100
})
pred_df.to_csv(os.path.join(RESULTS_DIR, 'predictions_actual_vs_predicted.csv'), index=False)
print(f"Predictions saved: {len(pred_df)} rows")

# Save metrics
metrics_df.to_csv(os.path.join(RESULTS_DIR, 'evaluation_metrics.csv'), index=False)
print("Metrics saved")

# Save cleaned dataset
df_fe.to_csv(os.path.join(RESULTS_DIR, 'cleaned_dataset.csv'), index=False)
print(f"Cleaned dataset saved: {df_fe.shape}")

print(f"\nAll outputs saved to: {RESULTS_DIR}")
print("Files:", os.listdir(RESULTS_DIR))

Predictions saved: 2188 rows
Metrics saved
Cleaned dataset saved: (16725, 61)

All outputs saved to: C:\Users\Oumaima\Desktop\Wiame\Resulta
Files: ['01_missing_values.png', '02_correlation_matrix.png', '03_time_series_eda.png', '04_distributions.png', '05_spectral_analysis.png', '06_stl_decomposition.png', '07_pca_analysis.png', '08_cluster_validation.png', '09_clusters.png', '10_training_curves.png', '11_predictions_scatter.png', '12_test_predictions.png', 'best_model.pth', 'cleaned_dataset.csv', 'evaluation_metrics.csv', 'predictions_actual_vs_predicted.csv']


---
## 8. Interpretation & Conclusions

### Model Performance
The LSTM-TFT hybrid model combines the sequential learning capability of LSTM networks with the attention mechanisms from Temporal Fusion Transformers. Key findings:

- **Variable Selection Network**: Automatically learns which input features are most informative at each timestep, providing interpretable feature importance
- **Huber Loss**: More robust to outliers than MSE, which is important given the heavy-tailed distribution of energy consumption across countries
- **Gated Residual Networks**: Allow the model to adaptively control information flow, preventing gradient degradation

### Key Drivers of Energy Demand
Based on the correlation analysis and PCA loadings:
1. **GDP** is the strongest predictor - economic output drives energy demand
2. **Population** scales energy needs at the national level
3. **Fossil fuel consumption** dominates the energy mix in most countries
4. **Historical consumption** (lag features) captures momentum and infrastructure inertia
5. **Energy intensity** (energy/GDP) reveals efficiency trends

### Statistical Insights
- **Stationarity**: Most country-level energy series are non-stationary (ADF), confirming the need for differencing or trend-aware models
- **Autocorrelation**: Strong autocorrelation (Ljung-Box) validates the use of sequential models (LSTM)
- **Spectral Analysis**: Dominant periods of 7-15 years suggest business/investment cycles in energy infrastructure
- **Clustering**: Countries naturally group by development stage and energy mix, with distinct consumption patterns

### Reliability of Long-term Forecasting
- Short-term (1-5 years): High reliability due to infrastructure inertia
- Medium-term (5-15 years): Moderate reliability - policy changes and technology shifts introduce uncertainty
- Long-term (15+ years): Low reliability - disruptive technologies, climate policies, and geopolitical shifts are unpredictable

### Limitations
1. **Annual granularity**: Cannot capture sub-annual (seasonal) patterns
2. **Missing data**: Many developing countries have sparse records
3. **Exogenous shocks**: Pandemics, wars, financial crises are not predictable from historical patterns
4. **Energy transition**: The ongoing shift to renewables may break historical patterns
5. **Aggregate data**: National-level data masks sectoral and regional variation

### Potential Improvements
1. Incorporate sub-annual (monthly/quarterly) data where available
2. Add exogenous variables: temperature, policy indicators, technology costs
3. Use probabilistic forecasting (quantile regression) for uncertainty quantification
4. Apply transfer learning across similar country clusters
5. Ensemble with traditional models (ARIMA, Prophet) for robustness